# ARIA Phase 5 — RAG Compliance Agent
### IE Business School × KPMG Spain · Corporate Capstone 2026

---

## What this notebook does

This notebook demonstrates the **RAG (Retrieval-Augmented Generation) Compliance Agent** built for the ARIA system.

Given any Airbnb listing, the agent:
1. Runs **rule-based compliance checks** against known thresholds (license presence, night caps, tax thresholds)
2. Performs **semantic search** (RAG) over a ChromaDB index of real regulations to retrieve the exact article that applies
3. Returns a **structured compliance verdict** including the article citation, violation detail, penalty exposure, and explanation

**Regulatory frameworks covered:**
- **Athens**: AMA framework (Law 4472/2017) — AADE registration requirement, 90-night cap, multi-property rules
- **Paris**: Loi Le Meur (Law 2024-1039) — 120-night cap, changement d'usage, tax regime changes

**Primary targets**: 137 Athens listings where `has_license == False` — identified in Phase 1 EDA, representing ~€1.03M annual revenue that would redistribute to compliant operators upon enforcement.

---

## Architecture

```
Regulation JSONs (AMA + Loi Le Meur)
        ↓ build_index.py
ChromaDB vector index  ←──── SentenceTransformer embeddings (all-MiniLM-L6-v2)
        ↓
compliance_agent.py
   ├── Rule checks  (license, night cap, tax threshold, multi-listing)
   └── RAG retrieval (semantic search → article citation)
        ↓
rag_compliance_v1.csv  (14,242 Athens listings × compliance fields)
        ↓
Phase 6 LangGraph node  (imported as check_compliance())
```

In [ ]:
# ── Standard imports ───────────────────────────────────────────────────────────
import os
import sys
import json
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

warnings.filterwarnings('ignore')

# Add project root to path so we can import from rag/
# The notebook lives inside rag/, so we go one level up
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Style
plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor':   '#1a1d27',
    'axes.edgecolor':   '#333',
    'axes.labelcolor':  '#ccc',
    'xtick.color':      '#888',
    'ytick.color':      '#888',
    'text.color':       '#eee',
    'grid.color':       '#333',
    'grid.alpha':       0.4,
    'figure.dpi':       120,
})

# Colour palette for risk levels (KPMG-aligned)
RISK_COLOURS = {
    'CRITICAL':  '#e63946',  # red
    'HIGH':      '#f4a261',  # orange
    'MEDIUM':    '#e9c46a',  # yellow
    'COMPLIANT': '#2a9d8f',  # teal
}

print('Imports OK')

---
## Section 1 — Load the ChromaDB Index and Agent

**Prerequisites:** Run `python rag/build_index.py` once from the project root before executing this notebook. The index is saved to `rag/chroma_db/` and loaded here.

In [ ]:
# Import the compliance agent from our module
from rag.compliance_agent import load_collection, check_compliance, run_batch

# Load the ChromaDB collection (loaded once, reused for all queries)
# This reads the vector index from rag/chroma_db/ and keeps it in memory
collection = load_collection()
print(f'ChromaDB collection loaded. Documents in index: {collection.count()}')

In [ ]:
# Display the full regulation knowledge base so we can see what's indexed
AMA_PATH = os.path.join(PROJECT_ROOT, 'rag', 'regulations_ama.json')
LE_MEUR_PATH = os.path.join(PROJECT_ROOT, 'rag', 'regulations_le_meur.json')

with open(AMA_PATH) as f:
    ama_articles = json.load(f)
with open(LE_MEUR_PATH) as f:
    le_meur_articles = json.load(f)

print('=== AMA Regulation Articles (Athens) ===')
for a in ama_articles:
    print(f"  [{a['article_id']}] {a['title']}")
    print(f"    Trigger: {a['compliance_trigger']} | Penalty: {a['penalty'][:80]}...")

print()
print('=== Loi Le Meur Articles (Paris) ===')
for a in le_meur_articles:
    print(f"  [{a['article_id']}] {a['title']}")
    print(f"    Trigger: {a['compliance_trigger']} | Penalty: {a['penalty'][:80]}...")

---
## Section 2 — Single Listing Demo

Walk through the agent step-by-step on two specific listings: one CRITICAL violation, one compliant.
This shows the full input → output pipeline that Phase 6 (LangGraph) will call.

In [ ]:
# Load the Athens dataset so we can pick real listings
ATHENS_PATH = os.path.join(PROJECT_ROOT, 'data', 'outputs', 'athens_predictions_v1.csv')
df_athens = pd.read_csv(ATHENS_PATH, low_memory=False)

# Derive has_license: True if the license field contains a real registration string
df_athens['has_license'] = df_athens['license'].apply(
    lambda x: False if (pd.isna(x) or str(x).strip().lower() in ('', 'nan', 'none', 'false', '0')) else True
)

print(f'Athens dataset: {len(df_athens):,} listings')
print(f'Unlicensed (has_license=False): {(~df_athens["has_license"]).sum()} listings')
print(f'Licensed:                       {df_athens["has_license"].sum()} listings')

In [ ]:
# ── Demo 1: An unlicensed listing (CRITICAL violation) ────────────────────────
# Pick the first unlicensed listing for a concrete example
unlicensed_row = df_athens[~df_athens['has_license']].iloc[0].to_dict()

print('INPUT LISTING:')
print(f"  listing_id:   {unlicensed_row['listing_id']}")
print(f"  city:         {unlicensed_row['city']}")
print(f"  neighbourhood:{unlicensed_row['neighbourhood']}")
print(f"  license:      {unlicensed_row['license']}")
print(f"  room_type:    {unlicensed_row.get('room_type', 'N/A')}")
print(f"  est. occupancy: {unlicensed_row.get('estimated_occupancy_l365d', 'N/A')}")
print(f"  est. revenue:   €{unlicensed_row.get('estimated_revenue_l365d', 'N/A'):,.0f}")
print()

# Run the compliance agent
result = check_compliance(unlicensed_row, collection)

print('COMPLIANCE RESULT:')
print(f"  compliant:       {result['compliant']}")
print(f"  risk_level:      {result['risk_level']}")
print(f"  violations:      {result['violation_count'] if 'violation_count' in result else len(result['violations'])}")
print(f"  primary_article: {result['primary_article']}")
print()
print('VIOLATIONS:')
for v in result['violations']:
    print(f"  [{v['severity']}] {v['article_id']}: {v['detail']}")
print()
print('REGULATION CITATION (top RAG result):')
print(f"  Article: [{result['rag_articles'][0]['article_id']}] {result['rag_articles'][0]['title']}")
print(f"  Penalty: {result['penalty_summary']}")
print()
print('EXPLANATION:')
print(f"  {result['explanation']}")

In [ ]:
# ── Demo 2: A compliant listing (has license, low occupancy) ──────────────────
# Pick a licensed listing with low estimated occupancy (well below the 90-night cap)
compliant_candidates = df_athens[
    df_athens['has_license'] &
    (df_athens['estimated_occupancy_l365d'] < 0.20) &  # < 73 nights/year
    (df_athens['host_total_listings_count'] < 3)
]
compliant_row = compliant_candidates.iloc[0].to_dict()

print('INPUT LISTING (compliant candidate):')
print(f"  listing_id:   {compliant_row['listing_id']}")
print(f"  neighbourhood:{compliant_row['neighbourhood']}")
print(f"  license:      {compliant_row['license']}")
print(f"  est. occupancy: {compliant_row.get('estimated_occupancy_l365d', 'N/A')} ({compliant_row.get('estimated_occupancy_l365d', 0)*365:.0f} nights)")
print()

result2 = check_compliance(compliant_row, collection)

print('COMPLIANCE RESULT:')
print(f"  compliant:   {result2['compliant']}")
print(f"  risk_level:  {result2['risk_level']}")
print(f"  violations:  {len(result2['violations'])}")
print()
print('EXPLANATION:')
print(f"  {result2['explanation']}")

---
## Section 3 — 137 Unlicensed Listings: The Primary Target List

The EDA (Phase 1) identified exactly 137 Athens listings where `has_license == False`. These are the ARIA compliance agent's primary targets — they are actively operating and pricing at market rate without a valid AADE registration number.

**Business context:** If these 137 listings were forced to exit the market through enforcement, the ~€1.03M annual revenue they generate would redistribute to compliant operators. For investors, these represent both enforcement risk (for the unlicensed listings) and market entry opportunity (for new compliant supply).

In [ ]:
# Load the pre-computed compliance output (generated by the batch run)
COMPLIANCE_PATH = os.path.join(PROJECT_ROOT, 'data', 'outputs', 'rag_compliance_v1.csv')
df_compliance = pd.read_csv(COMPLIANCE_PATH)

print(f'Full compliance output: {len(df_compliance):,} Athens listings')
print()
print('Risk level distribution:')
print(df_compliance['risk_level'].value_counts())

In [ ]:
# ── Focus on the 137 CRITICAL listings ───────────────────────────────────────
df_critical = df_compliance[df_compliance['risk_level'] == 'CRITICAL'].copy()

# Join back to Athens predictions to get revenue and pricing data
df_critical = df_critical.merge(
    df_athens[['listing_id', 'estimated_revenue_l365d', 'actual_price_eur',
               'predicted_price_eur', 'dist_zone', 'room_type']],
    on='listing_id', how='left'
)

total_revenue = df_critical['estimated_revenue_l365d'].sum()
median_price  = df_critical['actual_price_eur'].median()
median_rev    = df_critical['estimated_revenue_l365d'].median()

print(f'CRITICAL listings: {len(df_critical)}')
print(f'Total annual revenue at risk: €{total_revenue:,.0f}')
print(f'Median nightly price: €{median_price:.0f}')
print(f'Median annual revenue: €{median_rev:,.0f}')
print()
print('By neighbourhood (top 10):')
print(df_critical['neighbourhood'].value_counts().head(10).to_string())

In [ ]:
# ── Show a sample of the 137 with full compliance detail ─────────────────────
display_cols = [
    'listing_id', 'neighbourhood', 'risk_level', 'primary_article',
    'primary_detail', 'penalty_summary'
]

pd.set_option('display.max_colwidth', 80)
df_critical[display_cols].head(10)

---
## Section 4 — Visualisations

Four charts for the KPMG presentation:
1. **Risk level distribution** — full Athens market at a glance
2. **CRITICAL listings by neighbourhood** — where enforcement targets are concentrated
3. **Revenue at risk** — business impact per neighbourhood
4. **Article citation frequency** — which regulation articles are most triggered

In [ ]:
# ── Figure 1: Risk level distribution across all Athens listings ──────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('ARIA Compliance Agent — Athens STR Market Overview', fontsize=14, fontweight='bold', color='white', y=1.02)

risk_counts = df_compliance['risk_level'].value_counts()
# Reorder by severity
order = ['CRITICAL', 'HIGH', 'MEDIUM', 'COMPLIANT']
risk_counts = risk_counts.reindex([r for r in order if r in risk_counts.index])
colours = [RISK_COLOURS[r] for r in risk_counts.index]

# Bar chart
ax1 = axes[0]
bars = ax1.bar(risk_counts.index, risk_counts.values, color=colours, edgecolor='none', width=0.6)
for bar, val in zip(bars, risk_counts.values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
             f'{val:,}', ha='center', va='bottom', fontsize=11, color='white', fontweight='bold')
ax1.set_title('Athens Listings by Compliance Risk Level', color='white', pad=10)
ax1.set_ylabel('Number of Listings', color='#ccc')
ax1.set_ylim(0, risk_counts.max() * 1.15)
ax1.grid(axis='y', alpha=0.3)
ax1.set_facecolor('#1a1d27')

# Pie chart (percentages)
ax2 = axes[1]
wedges, texts, autotexts = ax2.pie(
    risk_counts.values,
    labels=risk_counts.index,
    colors=colours,
    autopct='%1.1f%%',
    startangle=90,
    pctdistance=0.8,
    textprops={'color': 'white', 'fontsize': 10}
)
for autotext in autotexts:
    autotext.set_color('white')
    autotext.set_fontsize(9)
ax2.set_title('Risk Level Share (14,242 listings)', color='white', pad=10)
ax2.set_facecolor('#1a1d27')

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'rag', 'fig_01_risk_distribution.png'),
            dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()
print('Saved: rag/fig_01_risk_distribution.png')

In [ ]:
# ── Figure 2: CRITICAL listings by neighbourhood ─────────────────────────────
crit_by_hood = df_critical['neighbourhood'].value_counts().head(12)

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(crit_by_hood.index[::-1], crit_by_hood.values[::-1],
               color=RISK_COLOURS['CRITICAL'], edgecolor='none')
for bar, val in zip(bars, crit_by_hood.values[::-1]):
    ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
            str(val), va='center', color='white', fontsize=10)

ax.set_title('CRITICAL Violations by Neighbourhood (137 Unlicensed Listings)',
             color='white', fontsize=13, pad=12)
ax.set_xlabel('Number of CRITICAL Listings', color='#ccc')
ax.set_xlim(0, crit_by_hood.max() * 1.15)
ax.grid(axis='x', alpha=0.3)
ax.set_facecolor('#1a1d27')

# Annotation
ax.text(0.98, 0.02,
        'Primary violation: AMA-Art-2\n(No AADE registration number)',
        transform=ax.transAxes, ha='right', va='bottom',
        fontsize=9, color='#e63946',
        bbox=dict(boxstyle='round', facecolor='#2a0a0a', alpha=0.8))

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'rag', 'fig_02_critical_by_neighbourhood.png'),
            dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()
print('Saved: rag/fig_02_critical_by_neighbourhood.png')

In [ ]:
# ── Figure 3: Revenue at risk — CRITICAL listings by neighbourhood ────────────
rev_by_hood = (
    df_critical.groupby('neighbourhood')['estimated_revenue_l365d']
    .sum()
    .sort_values(ascending=False)
    .head(12)
)

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(rev_by_hood.index[::-1], rev_by_hood.values[::-1] / 1000,  # convert to €k
               color='#f4a261', edgecolor='none')
for bar, val in zip(bars, rev_by_hood.values[::-1]):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'€{val/1000:.1f}k', va='center', color='white', fontsize=10)

ax.set_title('Annual Revenue at Risk by Neighbourhood\n(137 Unlicensed Athens Listings)',
             color='white', fontsize=13, pad=12)
ax.set_xlabel('Estimated Annual Revenue (€ thousands)', color='#ccc')
ax.set_xlim(0, rev_by_hood.max() / 1000 * 1.2)
ax.grid(axis='x', alpha=0.3)
ax.set_facecolor('#1a1d27')

total_k = rev_by_hood.sum() / 1000
ax.text(0.98, 0.02, f'Total at risk: €{total_k:.0f}k/year\n(top 12 neighbourhoods)',
        transform=ax.transAxes, ha='right', va='bottom',
        fontsize=9, color='#f4a261',
        bbox=dict(boxstyle='round', facecolor='#1a1200', alpha=0.8))

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'rag', 'fig_03_revenue_at_risk.png'),
            dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()
print('Saved: rag/fig_03_revenue_at_risk.png')

In [ ]:
# ── Figure 4: Article citation frequency (which rules are most triggered) ─────
# Count how many listings triggered each primary article across the full market
article_counts = df_compliance['primary_article'].value_counts()

# Colour each article bar by which city it belongs to
article_colours = [
    RISK_COLOURS['CRITICAL'] if 'AMA-Art-2' in a or 'AMA-Art-6' in a
    else RISK_COLOURS['HIGH'] if 'AMA-Art-3' in a
    else RISK_COLOURS['MEDIUM'] if 'AMA-Art-4' in a
    else RISK_COLOURS['COMPLIANT']
    for a in article_counts.index
]

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(article_counts.index, article_counts.values,
              color=article_colours, edgecolor='none', width=0.6)
for bar, val in zip(bars, article_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
            f'{val:,}', ha='center', va='bottom', color='white', fontsize=9)

ax.set_title('Regulation Article Citation Frequency — Athens Market\n(most triggered article per listing)',
             color='white', fontsize=13, pad=12)
ax.set_ylabel('Number of Listings', color='#ccc')
ax.set_xlabel('Article ID', color='#ccc')
ax.grid(axis='y', alpha=0.3)
ax.set_facecolor('#1a1d27')
plt.xticks(rotation=30, ha='right')

# Legend
patches = [
    mpatches.Patch(color=RISK_COLOURS['CRITICAL'], label='CRITICAL violation'),
    mpatches.Patch(color=RISK_COLOURS['HIGH'],     label='HIGH risk'),
    mpatches.Patch(color=RISK_COLOURS['MEDIUM'],   label='MEDIUM risk'),
    mpatches.Patch(color=RISK_COLOURS['COMPLIANT'],label='COMPLIANT'),
]
ax.legend(handles=patches, loc='upper right', facecolor='#1a1d27', labelcolor='white')

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'rag', 'fig_04_article_citations.png'),
            dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()
print('Saved: rag/fig_04_article_citations.png')

---
## Section 5 — Priority Target List: 137 Unlicensed × High-Risk

Cross-reference the 137 CRITICAL (unlicensed) listings with the Phase 3 LightGBM risk scores. Listings that are **both unlicensed AND high-risk** are the highest-priority enforcement targets — they are simultaneously violating registration law AND showing signs of operational decline.

For KPMG: these are the listings most likely to exit the market, creating the cleanest supply redistribution opportunity.

In [ ]:
# Load Phase 3 risk scores
RISK_PATH = os.path.join(PROJECT_ROOT, 'data', 'outputs', 'athens_risk_scores_v1.csv')
df_risk = pd.read_csv(RISK_PATH)

# Merge CRITICAL compliance listings with risk scores on listing_id
df_priority = df_critical.merge(df_risk, on='listing_id', how='inner')

print(f'CRITICAL compliance listings: {len(df_critical)}')
print(f'After joining with risk scores: {len(df_priority)}')
print()

# Priority tier: unlicensed AND high risk (risk_probability >= 0.70)
df_tier1 = df_priority[df_priority['high_risk_flag'] == 1]
df_tier2 = df_priority[df_priority['high_risk_flag'] == 0]

print(f'TIER 1 — Unlicensed + High Risk (top priority):  {len(df_tier1)} listings')
print(f'TIER 2 — Unlicensed + Moderate Risk:             {len(df_tier2)} listings')
print()
print(f'Tier 1 total annual revenue at risk: €{df_tier1["estimated_revenue_l365d"].sum():,.0f}')
print(f'Tier 2 total annual revenue at risk: €{df_tier2["estimated_revenue_l365d"].sum():,.0f}')

In [ ]:
# Display Tier 1 priority targets
tier1_display = df_tier1[[
    'listing_id', 'neighbourhood', 'risk_level',
    'primary_article', 'risk_probability', 'risk_band',
    'estimated_revenue_l365d', 'actual_price_eur'
]].sort_values('risk_probability', ascending=False)

tier1_display.columns = [
    'Listing ID', 'Neighbourhood', 'Compliance Risk',
    'Violated Article', 'Host Risk Prob.', 'Risk Band',
    'Est. Annual Revenue (€)', 'Nightly Price (€)'
]

print('TIER 1 PRIORITY TARGETS — Unlicensed & High Host Risk:')
tier1_display.head(15)

---
## Section 6 — Paris Demo (Loi Le Meur)

Demonstrate the agent on Paris listings. The Paris dataset is much larger (120k+) so we sample a representative set. The key Paris violation types are the 120-night cap and the revenue tax threshold.

In [ ]:
# Load Paris predictions
PARIS_PATH = os.path.join(PROJECT_ROOT, 'data', 'outputs', 'paris_predictions_v1.csv')
df_paris = pd.read_csv(PARIS_PATH, low_memory=False)

df_paris['has_license'] = df_paris['license'].apply(
    lambda x: False if (pd.isna(x) or str(x).strip().lower() in ('', 'nan', 'none', 'false', '0')) else True
)

print(f'Paris dataset: {len(df_paris):,} listings')
print(f'Unlicensed Paris:  {(~df_paris["has_license"]).sum():,}')
print(f'Licensed Paris:    {df_paris["has_license"].sum():,}')

if 'estimated_occupancy_l365d' in df_paris.columns:
    high_nights = df_paris['estimated_occupancy_l365d'] * 365 > 120
    print(f'Paris listings implying >120 nights: {high_nights.sum():,}')

In [ ]:
# Sample 500 Paris listings for a quick compliance scan
# (Full Paris run would take ~2 minutes — do it offline if needed)
sample_paris = df_paris.sample(500, random_state=42)

print('Running compliance agent on 500 Paris sample listings...')
paris_results = run_batch(sample_paris, collection)

df_paris_sample = pd.DataFrame([
    {
        'listing_id':    r['listing_id'],
        'city':          r['city'],
        'compliant':     r['compliant'],
        'risk_level':    r['risk_level'],
        'primary_article': r['primary_article'],
    }
    for r in paris_results
])

print()
print('Paris sample compliance distribution:')
print(df_paris_sample['risk_level'].value_counts())
print()
print('Most cited articles (Paris sample):')
print(df_paris_sample['primary_article'].value_counts().head(5))

---
## Section 7 — Phase 6 Integration Reference

This section documents exactly how the Phase 6 LangGraph orchestrator should call this agent. Copy this pattern into the compliance node.

In [ ]:
# ── LangGraph Node Template ────────────────────────────────────────────────────
# This is the exact pattern Phase 6 should use to call the compliance agent.
# The collection is loaded once at LangGraph graph initialisation,
# then passed into the node function on every call.

integration_template = '''
# ── In Phase 6 agents/ directory ──────────────────────────────────────────────

import sys, os
sys.path.insert(0, os.path.abspath(".."))  # project root

from rag.compliance_agent import load_collection, check_compliance

# Initialise once when the LangGraph graph is created
rag_collection = load_collection()

# ── LangGraph node function ────────────────────────────────────────────────────
def rag_compliance_node(state: dict) -> dict:
    """
    LangGraph compliance node.
    Input state must contain: listing (dict with listing metadata)
    Output: state updated with compliance_result key
    """
    listing = state["listing"]  # dict of listing fields (from mega dataset or CSV)

    # Run the compliance agent
    compliance_result = check_compliance(listing, rag_collection)

    # Add result to the LangGraph state so downstream nodes can use it
    return {
        **state,
        "compliance_result": compliance_result,
        "compliance_summary": (
            f"[{compliance_result[\'risk_level\']}] "
            f"{compliance_result[\'primary_article\']}: "
            f"{compliance_result[\'explanation\'][:200]}"
        )
    }
'''

print(integration_template)

---
## Section 8 — Output Summary

Final summary of all Phase 5 deliverables and their location.

In [ ]:
# ── Phase 5 output files ──────────────────────────────────────────────────────
files = {
    'rag/regulations_ama.json':              'AMA knowledge base — 7 articles (Athens)',
    'rag/regulations_le_meur.json':          'Loi Le Meur knowledge base — 7 articles (Paris)',
    'rag/build_index.py':                    'Builds ChromaDB vector index from JSONs',
    'rag/compliance_agent.py':               'Core agent — importable by Phase 6 LangGraph',
    'rag/ARIA_RAG_v1.ipynb':                 'This notebook — demo + visualisations',
    'rag/chroma_db/':                        'Vector index — NOT committed (in .gitignore)',
    'data/outputs/rag_compliance_v1.csv':    '14,242 Athens listings × compliance fields',
}

print('=== Phase 5 Deliverables ===')
for path, desc in files.items():
    full = os.path.join(PROJECT_ROOT, path)
    exists = '✓' if os.path.exists(full) else '✗ MISSING'
    print(f'  {exists}  {path}')
    print(f'       → {desc}')

print()
print('=== Key Numbers ===')
df_out = pd.read_csv(os.path.join(PROJECT_ROOT, 'data', 'outputs', 'rag_compliance_v1.csv'))
print(f'  Total Athens listings assessed:  {len(df_out):,}')
print(f'  CRITICAL (unlicensed — AMA-Art-2): {(df_out.risk_level=="CRITICAL").sum()}')
print(f'  HIGH (90-night cap — AMA-Art-3):   {(df_out.risk_level=="HIGH").sum():,}')
print(f'  MEDIUM (tax/multi-listing):        {(df_out.risk_level=="MEDIUM").sum():,}')
print(f'  COMPLIANT:                         {(df_out.risk_level=="COMPLIANT").sum():,}')
print()
df_crit_rev = df_critical['estimated_revenue_l365d'].sum()
print(f'  CRITICAL listings total revenue:   €{df_crit_rev:,.0f}/year')
print(f'  Phase 6 join key:                  listing_id')
print()
print('Phase 5 — COMPLETE')